[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/04_basic_usage/04_run_pipeline.ipynb)


# 04 — 파이프라인 실행·출력 구조

> 본문 [`04_basic_usage.md`](04_basic_usage.md) 와 **한 절씩 짝지어** 보세요.
> 이 노트북의 표·그래프·수치는 **여러분이 직접 돌린 결과**(`my_run/`)에서 계산합니다.
> **① 직접 설계 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 갑니다. 설계 셀은 NVIDIA GPU 전용이에요(CPU 폴백 없음) — Colab 이면 **런타임 → 런타임 유형 변경 → T4 GPU**.

## 0) 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `04_basic_usage/` 로 이동합니다. 로컬에서 `04_basic_usage/` 안에 열었다면 클론 없이 진행돼요.

In [ ]:
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # fork 했다면 본인 주소로 바꾸세요
CLONE_AS = "bio-guides"
CHAPTER  = "04_basic_usage"
PIP_PKGS = "pandas"   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HF 가중치 다운로드가 멈춘 채 매달리지 않도록 타임아웃을 겁니다.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):   # 클론 직후: cwd 아래로만 깊이 3까지
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")

# import 안 되는 패키지만 설치합니다.
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# 내 결과는 my_run/ 에 쌓이고, 없으면 커밋된 레퍼런스로 폴백합니다.
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """my_run/ 에서 먼저 찾고, 없으면 레퍼런스 폴더에서 찾습니다."""
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 돌린 설계 결과를 이어받습니다(없으면 레퍼런스로 폴백)."""
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 레퍼런스 그림을 덮어쓰지 않도록 my_ 접두어.
def my_fig(name):
    return f"my_{name}"

print("작업 폴더 :", pathlib.Path.cwd())

## 1) `boltzgen run` 명령 구조 — 두 숫자가 전부를 좌우해요 (본문 4.1·4.3)

```
boltzgen run <명세.yaml> --output <폴더> --protocol <프로토콜> --num_designs <중간 수> --budget <최종 선별 수>
```

`--protocol` 이 내부 설정을 통째로 정합니다. `protein-anything` 이면 **6스텝** —
design → inverse_folding → folding → design_folding → analysis → filtering.

나머지 둘, `--num_designs` 와 `--budget` 이 품질과 실행 시간을 가장 크게 좌우해요. 기준부터 잡고 갑니다.

In [ ]:
run_cmd = ("boltzgen run example/vanilla_protein/1g13prot.yaml "
           "--output workbench/vanilla --protocol protein-anything "
           "--num_designs 1000 --budget 50")
print(run_cmd)

print("\n--num_designs — 표본을 많이 뽑을수록 좋은 꼬리를 만납니다 (본문 4.3)")
for case, rec in [("빠른 테스트 · 환경 검증", "4 ~ 100"),
                  ("일반 설계",              "1,000 ~ 5,000"),
                  ("어려운 타깃 · 복잡한 제약", "10,000 ~ 60,000")]:
    print(f"   {case:22s} → {rec}")
print("   시간·계산이 비례해 늘어나요. 작게 테스트한 뒤 크게 프로덕션으로 갑니다.")

print("\n--budget — 점수 상위 N개가 아니라, 서로 다른 전략까지 골라 담는 최종 후보 수 (본문 4.3)")
for proto, b in [("peptide-anything", 30), ("protein-small_molecule", 40), ("protein-anything", 50)]:
    print(f"   {proto:24s} --budget {b}")
print("   본문 4.7 실행 예시의 값이에요. 1~3등이 거의 같은 서열이면 하나가 실패할 때 셋 다 실패하니,")
print("   다양성을 섞는 게 실험 성공 확률을 높이는 보험입니다.")

print("\n아래 실습은 이 표의 맨 윗줄, num_designs 4 · budget 2 스모크 규모로 갑니다.")

## 2) 실행 중 무엇을 보며 기다리나 (본문 4.8)

다음 셀은 **5분 동안 로그만 쏟아냅니다.** 무엇을 보며 기다릴지 먼저 정해두면
"멈춘 건지 도는 건지"로 헷갈리지 않아요.

In [ ]:
import shutil, subprocess

print("로그에서 이 세 줄만 쫓으면 됩니다 (본문 4.8)")
print("   Pipeline step k of N                    ← k 번째 스텝 시작")
print("   Step ... completed successfully in Xs   ← 그 스텝 정상 종료")
print("   Traceback                               ← 여기서 멈췄다면 위로 스크롤해 원인 확인")
print("   k 가 1 → 2 → 3 으로 올라가면 정상. 같은 k 에서 오래 머무르면 그 스텝이 무거운 거예요.")

print("\n터미널에서 돌릴 때 (본문 4.8)")
print("   watch -n 1 nvidia-smi                       # 다른 터미널에서 GPU 사용량")
print("   boltzgen run ... > run.log 2>&1 &")
print("   tail -f run.log                             # 로그 추적")

print("\n지금 이 런타임의 GPU 상태")
if shutil.which("nvidia-smi"):
    q = "name,memory.used,memory.total,utilization.gpu"
    out = subprocess.run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    print("  ", out.replace("\n", "\n   "))
    print("   설계가 돌면 memory.used 가 수 GB 로 올라가고 utilization 이 높게 유지돼요.")
    print("   0 % 에서 오래 머물면 가중치(~6GB) 를 내려받는 중일 수 있습니다.")
else:
    print("   nvidia-smi 없음 → CPU 런타임이라 아래 설계 셀은 스스로 건너뜁니다.")
    print("   직접 돌리려면 [런타임 → 런타임 유형 변경 → T4 GPU] 로 바꾸세요.")

## 3) 직접 돌려보기 — 내 결과 만들기

- 학습용 규모 `num_designs=4 --budget=2` (가장 작은 스모크 규모 — 6스텝이 전부 도는지 확인하는 게 목적)
- 소요 시간 실측 **307초**(최종 2개) — **24GB GPU · 가중치 캐시** 기준이에요. Colab T4 는 가속 커널이 꺼져 더 걸리고(T4 실측치 없음), 첫 실행은 가중치 ~6GB 다운로드가 더 붙어요.
- 건너뛰면 아래 분석이 커밋된 레퍼런스 결과로 이어집니다.

In [ ]:
SPEC, PROTOCOL = "example/vanilla_protein/1g13prot.yaml", "protein-anything"
NUM_DESIGNS, BUDGET = 4, 2

import shutil
OUT = MY.resolve()

def _gpu():
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return shutil.which("nvidia-smi") is not None

if not _gpu():
    print("GPU 런타임이 아니라 설계 실행을 건너뜁니다 — 아래 분석은 레퍼런스 결과로 이어집니다.")
    print("  · 직접 돌리려면 Colab [런타임 → 런타임 유형 변경 → T4 GPU] 후 이 셀을 다시 실행하세요.")
else:
    SRC = ADV_ROOT / ".boltzgen_src"          # 예제 명세·타깃 CIF 가 들어 있는 BoltzGen 소스
    if not SRC.exists():
        _run(f'git clone --depth 1 https://github.com/HannesStark/boltzgen.git "{SRC}"')
    if not _have("boltzgen"):
        _run(f'"{sys.executable}" -m pip -q install -e "{SRC}"')
    try:
        _run(f'cd "{SRC}" && boltzgen run {SPEC} --output "{OUT}" --protocol {PROTOCOL} '
             f'--num_designs {NUM_DESIGNS} --budget {BUDGET}')
        print("\n내 결과 →", OUT / "final_ranked_designs")
    except Exception as e:
        print("\n설계 실행이 도중에 멈췄어요 —", type(e).__name__)
        print("  · 이 셀을 다시 실행하면 같은 --output 산출물을 재사용해 이어서 끝냅니다(실측 763초 → 재개 486초).")
        print("  · GPU 메모리가 부족했다면 NUM_DESIGNS 4, BUDGET 2 로 줄이세요.")

## 4) 방금 돌아간 스텝을 `steps.yaml` 로 되짚기 (본문 4.2)

로그는 지나갔으니 이제 산출물 쪽에서 확인해요. **어떤 스텝이 실제로 실행됐는지는 출력 루트의 `steps.yaml`** 에
그대로 적혀 있고, 스텝마다 결과를 놓는 폴더가 정해져 있습니다.

설계를 건너뛰었다면 Ch.05 의 레퍼런스 결과(`05_result_interpretation/data/vanilla`)가 해부 대상이 됩니다 —
`--output` 폴더 구조는 어느 쪽이든 같아요.

In [ ]:
import re
REF = "../05_result_interpretation/data/vanilla"

steps    = find_one("steps.yaml", REF)
OUT_ROOT = steps.parent                       # steps.yaml 이 있는 곳 = --output 루트
names    = re.findall(r"-\s*name:\s*(\S+)", steps.read_text())

WHAT = {
    "design":          ("타깃에 맞는 백본 생성",  "intermediate_designs/"),
    "inverse_folding": ("백본에 서열 채우기",    "intermediate_designs_inverse_folded/"),
    "folding":         ("바인더+타깃 재접힘",    "intermediate_designs_inverse_folded/refold_cif/"),
    "design_folding":  ("바인더 단독 재접힘",    "intermediate_designs_inverse_folded/refold_design_cif/"),
    "affinity":        ("소분자 친화도 예측",    "intermediate_designs_inverse_folded/"),
    "analysis":        ("메트릭 계산",          "intermediate_designs_inverse_folded/*_metrics_analyze.csv"),
    "filtering":       ("하드필터 + 다양성 선별", "final_ranked_designs/"),
}

print("출력 루트   :", OUT_ROOT)
print("실행된 스텝 :", len(names), "개\n")
for i, nm in enumerate(names, 1):
    what, out = WHAT.get(nm, ("-", "-"))
    print(f"  [{i}] {nm:16s} {what:14s} → {out}")

print("\nprotein-anything 은 6스텝이에요. 펩타이드·나노바디·항체는 design_folding 이 빠져 5스텝,")
print("소분자는 affinity 가 붙어 7스텝으로 늘어납니다 (본문 4.2·4.7).")

## 5) 출력 트리 실제로 열어보기 (본문 4.6)

스텝마다 폴더가 정해져 있다고 했으니, 그 폴더가 진짜 생겼는지 확인할 차례예요.
본문 4.6 의 트리를 그대로 체크리스트로 만들어 **있음/없음**을 표시하고, 이어서 안에 뭐가 들었는지 셉니다.

In [ ]:
from collections import Counter

EXPECT = [
    ("config/",                                              "스텝별 설정 yaml (configure 단계 산출)"),
    ("steps.yaml",                                           "실행된 스텝 매니페스트"),
    ("intermediate_designs/",                                "[design] 백본 cif + 메타데이터 npz"),
    ("intermediate_designs_inverse_folded/",                 "[inverse_folding] 서열이 채워진 cif"),
    ("intermediate_designs_inverse_folded/refold_cif/",      "[folding] 재접힘 복합체 — 분석의 진짜 입력"),
    ("intermediate_designs_inverse_folded/refold_design_cif/", "[design_folding] 바인더 단독 재접힘"),
    ("final_ranked_designs/",                                "[filtering] 최종셋 · 메트릭 CSV · results_overview.pdf"),
]
print("본문 4.6 트리 대조\n")
for rel, note in EXPECT:
    p = OUT_ROOT / rel.rstrip("/")
    if p.is_dir():
        print(f"  [O] {rel:56s} {sum(1 for _ in p.glob('*')):5d}개  {note}")
    elif p.is_file():
        print(f"  [O] {rel:56s} {p.stat().st_size:5d}B  {note}")
    else:
        print(f"  [ ] {rel:56s}    -    {note}")

print("\n실제로 들어 있는 것 (하위 폴더별 파일 수·확장자)\n")
seen = {}
for f in OUT_ROOT.rglob("*"):
    if f.is_file():
        key = f.parent.relative_to(OUT_ROOT).as_posix() or "."
        seen.setdefault(key, []).append(f.suffix or "(확장자없음)")
for key in sorted(seen):
    ext = ", ".join(f"{e} {n}" for e, n in Counter(seen[key]).most_common(4))
    print(f"  {key + '/':56s} {len(seen[key]):5d}개  {ext}")

print("\n빈 칸은 저장소에 커밋하지 않은 중간 산출물이에요(용량 문제) — 레퍼런스에는 최종 산출물만 평평하게 담겨 있어요.")
print("직접 돌린 my_run/ 에는 위 일곱 줄이 모두 생깁니다.")

## 6) 최종 산출물 — 순위 붙은 CIF 와 메트릭 CSV (본문 4.6)

트리에서 `final_ranked_designs/` 를 확인했으니, 그 안의 최종 결과물을 직접 엽니다.
폴더·파일 이름에 `budget` 이 박히므로(`final_<budget>_designs/`, `final_designs_metrics_<budget>.csv`)
글롭으로 찾아요.

In [ ]:
import pandas as pd

csv  = find_one("final_designs_metrics_*.csv", REF)
base = csv.parent
cifs = sorted(base.glob("final*designs/*.cif"))

print("최종 디자인 CIF — 순위 붙은 단일 파일 (서열·구조 내장)")
for p in cifs[:5]:
    print("   ", p.parent.name + "/" + p.name)
print("    ... 총", len(cifs), "개")

print("\n메트릭 CSV")
for f in sorted(base.glob("*metrics*.csv")):
    print(f"    {f.name:38s} ({f.stat().st_size} bytes)")

df = pd.read_csv(csv)
print("\n최종셋 디자인", len(df), "개 | 컬럼", df.shape[1], "개 — 한 디자인이 한 행이에요")
df[cols_in(df, "id", "final_rank", "design_ptm", "design_to_target_iptm",
           "filter_rmsd")].sort_values("final_rank")

판정 — `final_rank` 가 1부터 `budget` 까지 빠짐없이 있고 pTM·ipTM·RMSD 가 비어 있지 않으면
6스텝이 정상 완주한 거예요. 한 스텝이 죽어 CSV 가 안 나왔다면 **같은 `--output` 으로 같은 명령을 다시** 실행하세요 —
이미 만들어진 산출물을 재사용해 이어서 끝냅니다(본문 4.7).

기억해 둘 것 두 가지(본문 4.6).
- 인터페이스 분석·시각화는 **`refold_cif/` 또는 최종 CIF 기준**. inverse_folded 직후 파일은 측쇄가 원점에 뭉쳐 있어요.
- `all_designs_metrics.csv` 는 선별 **전** 전체, `final_designs_metrics_<budget>.csv` 는 선별 **후** 최종셋이에요.

## 7) 다시 돌릴 때 — `--steps` 부분 실행 (본문 4.4)

방금 표를 보고 "선별 기준을 바꿔 다시 뽑고 싶다"는 생각이 들었다면, 전체를 처음부터 돌릴 필요가 없어요.

In [ ]:
spec_path = globals().get("SPEC", "spec.yaml")

print("사용 가능한 스텝 7종 (본문 4.4)")
print("   design · inverse_folding · design_folding · folding · affinity · analysis · filtering")
print("   affinity 는 소분자 프로토콜에서만 붙습니다 (본문 4.7 · Ch.10).")

print("\n# 앞 절반만 — 백본과 서열까지")
print(f"boltzgen run {spec_path} --output out --steps design inverse_folding")
print("\n# 분석이 끝난 결과에서 선별만 다시 — 몇 초면 끝납니다")
print(f'boltzgen run {spec_path} --output "{OUT_ROOT}" --steps filtering --budget 3')

print("\n핵심 패턴 — 무거운 design~analysis 는 한 번만, filtering 은 기준을 바꿔가며 여러 번 (본문 4.4).")
print("바꿔볼 기준은 --budget, --metrics_override, --additional_filters 예요. 실험은 Ch.06 에서 이어갑니다.")

## 8) 스텝 안까지 조절 — `--config` 와 실행 제어 옵션 (본문 4.5)

규모를 키우면 이번엔 자원이 문제가 돼요. 스텝 내부 설정은 `--config <스텝명> 키=값` 으로 덮어씁니다.

In [ ]:
print("--config <스텝명> 키=값 — 그 스텝의 내부 설정만 덮어씁니다 (본문 4.5)")
print("boltzgen run spec.yaml --output out --config folding num_workers=4 trainer.devices=2")

print("\n자주 쓰는 실행 제어 옵션 (본문 4.5)")
opts = [
    ("--devices N",                    "사용할 GPU 수",            "여러 장으로 나눠 돌릴 때"),
    ("--num_workers N",                "데이터로더 워커 수",        "CPU 는 노는데 GPU 가 굶을 때"),
    ("--use_kernels auto|true|false",  "CUDA 가속 커널 (기본 auto)", "커널 로딩 오류를 우회할 때 (Ch.03)"),
    ("--reuse",                        "기존 결과 재사용",          "같은 폴더에 디자인을 더 얹을 때"),
    ("--diffusion_batch_size N",       "디자인 배치 크기",          "VRAM 이 모자라면 1 로"),
    ("--inverse_fold_num_sequences N", "백본당 서열 수",           "백본은 두고 서열만 더 볼 때"),
    ("--inverse_fold_avoid 'KEC'",     "특정 잔기 금지",           "Cys 등을 피한 설계가 필요할 때"),
]
for opt, mean, when in opts:
    print(f"   {opt:32s} {mean:20s} | {when}")

print("\n실습 규모에서 바로 쓰는 조합")
print("   # VRAM 이 좁은 카드(T4 등)에서 100개 이상 뽑을 때")
print("   boltzgen run spec.yaml --output out --num_designs 100 --budget 10 --diffusion_batch_size 1")
print("   # bf16 미지원 카드에서 정밀도 오류가 날 때 (Ch.03)")
print("   boltzgen run spec.yaml --output out --config folding trainer.precision=32")

### 이 챕터에서 확인한 것

`--num_designs`·`--budget` 의 기준을 잡고, 스모크 규모로 6스텝을 완주시켜 `steps.yaml` 부터
`final_ranked_designs/` 까지 출력 트리를 직접 열어봤어요. 다시 돌릴 땐 `--steps` 로 필요한 스텝만,
자원이 모자라면 `--config` 로 스텝 안까지 조절하면 됩니다.

이제 손에 남은 건 `final_designs_metrics_<budget>.csv` 한 장이에요. 그 안의 200여 개 컬럼이 무엇을 재고
**순위가 어떻게 정해지는지**는 Ch.05 에서 데이터로 확인합니다.